# Ethereum Gas Fee Study

**Question:** When is the cheapest time to send a transaction on Ethereum, how much could a company save by timing it, and what causes fee spikes?

**Data:** hourly summary of Ethereum blocks, Jan to Sep 2026, pulled from BigQuery with [`sql/hourly_gas.sql`](../sql/hourly_gas.sql) and saved as `data/gas_hourly.csv`.

## Step 2: Check the data

Load the hourly data and convert timestamps from UTC to New York time, so findings read as "weekday afternoons" rather than UTC hours.

In [1]:
import pandas as pd

df = pd.read_csv("../data/gas_hourly.csv")
df["hour"] = pd.to_datetime(df["hour"], utc=True).dt.tz_convert("America/New_York")
df.head()

,hour,blocks,avg_base_fee_gwei,tx_count,block_fullness
0,2025-12-31 19:00:00-05:00,299,0.035638,76759,0.506983
1,2025-12-31 20:00:00-05:00,300,0.030256,68399,0.511707
2,2025-12-31 21:00:00-05:00,299,0.031566,66667,0.512734
3,2025-12-31 22:00:00-05:00,297,0.031615,72503,0.505298
4,2025-12-31 23:00:00-05:00,299,0.029470,69984,0.510130


Run the checks. Two facts about Ethereum tell us what to expect: about one block every 12 seconds (about 300 per hour), and 9 months is about 6,550 hours.

In [2]:
print(len(df))                    # about 6,550?
print(df["blocks"].describe())    # mostly near 300?
print(df.isna().sum())            # anything missing?
print(df["avg_base_fee_gwei"].describe())

6552
count    6552.000000
mean      298.791972
std         1.182962
min       288.000000
25%       298.000000
50%       299.000000
75%       300.000000
max       300.000000
Name: blocks, dtype: float64
hour                 0
blocks               0
avg_base_fee_gwei    0
tx_count             0
block_fullness       0
dtype: int64
count    6552.000000
mean        0.267236
std         0.715701
min         0.026319
25%         0.058979
50%         0.111527
75%         0.195360
max        16.785048
Name: avg_base_fee_gwei, dtype: float64


### Data checks

- **Row count:** 6,552 hours, which is exactly 273 days × 24 hours from Jan 1 to Sep 30, 2026. No hours are missing.
- **Blocks per hour:** median 299, range 288 to 300. No suspicious hours, which fits one block every 12 seconds.
- **Missing values:** none in any column.
- **Fee range:** min 0.026 gwei, median 0.11 gwei, max 16.8 gwei. The max is about 150 times the median. These are real spikes to explain later, not errors to delete. The biggest ones cluster around Feb 5 and 6 and Sep 24.
- **Time zone:** converted from UTC to New York time so patterns read in business terms (for example "weekday afternoons") for a US company.